In [7]:
# !pip show fastai
# !pip show torch


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
from fastai.vision.all import *
from PIL import Image as PIL_Original

# -- Step 1: Patch PILImage.create for dummy images --
# def fake_open_image(fn):
#     dummy_tensor = torch.zeros(3, 224, 224).byte()
#     dummy_tensor = dummy_tensor.permute(1, 2, 0)
#     import PIL.Image
#     pil_img = PIL.Image.fromarray(dummy_tensor.numpy())
#     return PILImage(pil_img)

#PILImage.create = fake_open_image  # Use fake image loader

# -- Step 2: Dummy DataFrame --
dummy_data = {
    'image': ['714'] * 5,
    'level': [0, 1, 2, 3, 4]
}
dummy_df = pd.DataFrame(dummy_data)

# -- Step 3: Create dls --
dls = ImageDataLoaders.from_df(
    dummy_df,
    path='/content',
    fn_col='image',
    label_col='level',
    suff='.jpg',
    valid_pct=0.0,
    bs=1,
    item_tfms=[],
    batch_tfms=[],
    shuffle=False
)

In [10]:
dls.c

5

In [11]:
dls.vocab

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

In [12]:
from fastai.vision.all import *
from fastcore.foundation import L
import itertools

# Patch missing 'starmap' method in fastcore's L class
if not hasattr(L, 'starmap'):
    L.starmap = lambda self, f: L(itertools.starmap(f, self))

import torch
import torchvision.models as models

learn = vision_learner(dls, resnet34, path='.',
    loss_func=FocalLoss(),
    metrics=[accuracy],  n_out=5)

# Load weights with map_location in torch.load
checkpoint = torch.load('/content/drive/MyDrive/fundusnap/retinopathy201519-rn34-1.pth', weights_only=False, map_location=torch.device('cpu'))

# checkpoint is a dict with keys: 'model', 'opt', etc.
state_dict = checkpoint['model']  # extract just the model weights

learn.model.load_state_dict(state_dict)

<All keys matched successfully>

In [13]:
print(learn.model)


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [14]:
learn.dls.vocab = [0,1,2,3,4]  # or whatever your classes are


In [15]:
img_path = '/content/714.jpg'  # your image path

pred_class, pred_idx, probs = learn.predict(img_path)
probs = F.softmax(probs, dim=0)
print(probs)

probs_softmax = F.softmax(probs, dim=0)
print(probs_softmax)
print(f"Sum of probs: {probs_softmax.sum().item()}")  # Should be 1.0
max_prob, max_idx = probs_softmax.max(0)
print(f"Prediction idx: {max_idx.item()}, Probability: {max_prob.item():.4f}")

tensor([0.0833, 0.3694, 0.3070, 0.2251, 0.0152])
tensor([0.1764, 0.2348, 0.2206, 0.2033, 0.1648])
Sum of probs: 1.0
Prediction idx: 1, Probability: 0.2348


In [16]:
# !pip install onnxscript onnx

In [17]:
import torch
import torch.nn as nn

# Set the model to evaluation mode
learn.model.eval()

# Example dummy input (adjust size to your model input, e.g., 3x224x224)
dummy_input = torch.randn(1, 3, 224, 224)

# Workaround: PyTorch's ONNX exporter doesn't support AdaptiveMaxPool2d.
# Fastai uses AdaptiveConcatPool2d (which contains AdaptiveMaxPool2d and AdaptiveAvgPool2d).
# We replace it with an ONNX-friendly version that performs global max and average pooling.
class ONNXFriendlyConcatPool(nn.Module):
    def forward(self, x):
        # Global max pooling
        max_pool = x.amax(dim=[-1, -2], keepdim=True)
        # Global average pooling
        avg_pool = x.mean(dim=[-1, -2], keepdim=True)
        # Fastai typically concatenates max and avg
        return torch.cat([max_pool, avg_pool], dim=1)

# Replace the incompatible layer (usually the first layer of the head in fastai vision models)
learn.model[1][0] = ONNXFriendlyConcatPool()

# Export to ONNX
torch.onnx.export(learn.model, dummy_input, "model.onnx",
                  export_params=True, opset_version=14,
                  do_constant_folding=True,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'},
                                'output': {0: 'batch_size'}})
print("Model successfully exported to model.onnx")

/tmp/ipykernel_7309/580617149.py:26: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(learn.model, dummy_input, "model.onnx",
W0731 02:19:13.456000 7309 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 73 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully exported to model.onnx


# Installing Requirements for ONNX-TF2

In [18]:
# !pip install onnx-tf

In [19]:
# !pip install onnx2tf

In [20]:
# !pip install onnx-graphsurgeon --extra-index-url https://pypi.nvidia.com

In [21]:
# !pip install ai-edge-litert

In [22]:
# !python3 -m pip install onnx_graphsurgeon --index-url https://pypi.ngc.nvidia.com
# !pip install sng4onnx

# Converting ONNX to Tensorflow

In [23]:
!onnx2tf --input_onnx_file_path model.onnx --output_folder_path model_tf


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                    ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add                │ 16             │ 16               │
│ BatchNormalization │ 2              │ 2                │
│ Concat             │ 2              │ 2                │
│ Constant           │ 84             │ 84               │
│ Conv               │ 36             │ 36               │
│ Gemm               │ 2              │ 2                │
│ MaxPool            │ 1              │ 1                │
│ ReduceMax          │ 1              │ 1                │
│ ReduceMean         │ 1              │ 1                │
│ Relu               │ 34             │ 34               │
│ Reshape            │ 1              │ 1                │
│ Shape              │ 1              │ 